# Lesson 02: 충돌과 바닥 - 공이 바닥에 부딪혀 튕기기

## 학습 목표
1. 고정된 바닥(floor) 물체 만들기
2. 충돌(collision) 시스템 활성화하기
3. 접촉 재질(contact material) 설정하기
4. 반발 계수(restitution)로 튕김 정도 조절하기

## 새로 배우는 Chrono API
| 클래스/메서드 | 역할 |
|---|---|
| `SetCollisionSystemType(Type_BULLET)` | Bullet 충돌 감지 엔진 활성화 |
| `ChContactMaterialNSC()` | 접촉 재질 생성 (마찰, 반발) |
| `SetFriction()` | 마찰 계수 (0=미끄러움, 1=거침) |
| `SetRestitution()` | 반발 계수 (0=찰흙, 1=완전탄성) |
| `SetFixed(True)` | 물체 고정 (바닥 등) |
| `EnableCollision(True)` | 물체의 충돌 활성화 |
| `AddCollisionShape()` | 충돌 형태 추가 |
| `ChCollisionShapeBox(mat, x, y, z)` | 상자형 충돌체 |
| `ChCollisionShapeSphere(mat, r)` | 구형 충돌체 |
| `SetInertiaXX()` | 관성 모멘트 대각 성분 설정 |

In [ ]:
import pychrono as chrono

## 1단계: 시스템 + 충돌 엔진 설정

- Bullet = 오픈소스 물리 엔진의 충돌 감지 알고리즘
- 충돌이 필요한 시뮬레이션에서는 반드시 설정해야 함

In [ ]:
sys = chrono.ChSystemNSC()
sys.SetGravitationalAcceleration(chrono.ChVector3d(0, -9.81, 0))
sys.SetCollisionSystemType(chrono.ChCollisionSystem.Type_BULLET)

print("시스템 + Bullet 충돌 엔진 설정 완료")

## 2단계: 접촉 재질 만들기

두 물체가 부딪힐 때 어떻게 반응할지 결정하는 속성:
- **마찰 계수 (friction)**: 0=완전 미끄러움, 1=매우 거침
- **반발 계수 (restitution)**: 0=안 튕김(찰흙), 1=완전 튕김(슈퍼볼)

> 두 물체가 만나면 각각의 재질이 합산되어 최종 접촉 특성이 결정됨

In [ ]:
material = chrono.ChContactMaterialNSC()
material.SetFriction(0.3)
material.SetRestitution(0.7)     # 꽤 잘 튕김

print(f"마찰 계수: 0.3")
print(f"반발 계수: 0.7")

## 3단계: 바닥 만들기

바닥 = 고정된(Fixed) 물체 + 충돌 형태

**충돌이 작동하려면 두 가지가 모두 필요:**
1. `EnableCollision(True)` — 충돌 감지 활성화
2. `AddCollisionShape(...)` — 충돌 형태 정의

In [ ]:
floor = chrono.ChBody()
floor.SetMass(1)                                # 고정체라 무관하지만 필요
floor.SetPos(chrono.ChVector3d(0, -1, 0))       # 바닥면이 y=0이 되도록
floor.SetFixed(True)                            # 고정! 중력에 안 떨어짐
floor.EnableCollision(True)

# 충돌 형태: 20m x 2m x 20m 상자
floor.AddCollisionShape(chrono.ChCollisionShapeBox(material, 20, 2, 20))
sys.AddBody(floor)

print(f"바닥 위치: y = {floor.GetPos().y} m, 고정: True")

## 4단계: 공 만들기

관성 모멘트(Inertia):
- 구의 공식: `I = 2/5 * m * r²`
- `SetInertiaXX()`에 대각 성분 (Ixx, Iyy, Izz) 설정
- 구는 모든 축에 대해 동일

In [ ]:
ball = chrono.ChBody()
ball.SetMass(1.0)
ball.SetPos(chrono.ChVector3d(0, 5, 0))         # 높이 5m
ball.SetFixed(False)
ball.EnableCollision(True)

# 충돌 형태: 반지름 0.3m 구
radius = 0.3
ball.AddCollisionShape(chrono.ChCollisionShapeSphere(material, radius))

# 관성 모멘트 설정 (구: I = 2/5 * m * r²)
inertia = 2.0 / 5.0 * ball.GetMass() * radius**2
ball.SetInertiaXX(chrono.ChVector3d(inertia, inertia, inertia))

sys.AddBody(ball)

print(f"공: 질량={ball.GetMass()} kg, 반지름={radius} m")
print(f"관성 모멘트: {inertia:.4f} kg·m²")

## 5단계: 시뮬레이션 실행 - 바운스 감지

- `dt = 0.001` (1ms) — 충돌 정확도를 위해 Lesson 01보다 작게
- 바운스 감지: 속도가 음→양으로 바뀌는 순간

In [ ]:
dt = 0.001       # 충돌 정확도를 위해 작은 시간 간격
end_time = 4.0

print(f"{'시간(s)':>8}  {'높이(m)':>10}  {'속도(m/s)':>10}  상태")
print("─" * 50)

prev_vy = 0
bounce_count = 0

while sys.GetChTime() < end_time:
    sys.DoStepDynamics(dt)

    t = sys.GetChTime()
    y = ball.GetPos().y
    vy = ball.GetPosDt().y

    # 바운스 감지: 속도가 음→양
    if prev_vy < -0.5 and vy > 0:
        bounce_count += 1
        print(f"{t:8.3f}  {y:10.4f}  {vy:10.4f}  *** 바운스 #{bounce_count}! ***")

    elif abs(t % 0.5) < dt * 0.5 or abs(t % 0.5 - 0.5) < dt * 0.5:
        status = "낙하 중" if vy < -0.1 else ("상승 중" if vy > 0.1 else "정지")
        print(f"{t:8.3f}  {y:10.4f}  {vy:10.4f}  {status}")

    prev_vy = vy

In [ ]:
print(f"총 바운스 횟수: {bounce_count}")
print(f"최종 높이: {ball.GetPos().y:.4f} m")
print(f"최종 속도: {ball.GetPosDt().y:.4f} m/s")

## 실험해보기

1. **반발 계수를 바꾸면?** `SetRestitution(0.95)` → 슈퍼볼처럼 높이 튕김
2. **반발 계수 0이면?** `SetRestitution(0.0)` → 찰흙처럼 안 튕김
3. **초기 높이를 바꾸면?** 높을수록 첫 바운스 속도가 빠름
4. **dt를 크게 하면?** `dt = 0.01` → 충돌 감지가 부정확해짐 (관통 가능)